# Phase 1 LayoutLMv3 字段抽取训练

本 Notebook 的前半部分用于 Phase 1A：SROIE OCR + Regex baseline smoke 闭环。fixture smoke result 只能证明流程可运行，不能作为简历指标。

## 1. 环境检查

确认当前 Python 能导入项目代码。PaddleOCR、torch、transformers 是 Phase 1 可选依赖，正式 OCR 或训练时再安装。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('project_root=', PROJECT_ROOT)


## 2. 路径配置

默认读取仓库里的最小 fixture。真实 SROIE 数据请放到 `data/phase1/sroie/raw/train` 或 `data/phase1/sroie/raw/test`。

In [ ]:
FIXTURE_RAW = PROJECT_ROOT / 'tests' / 'fixtures' / 'sroie_minimal' / 'raw'
FIXTURE_PROCESSED = PROJECT_ROOT / 'tests' / 'fixtures' / 'sroie_minimal' / 'processed.jsonl'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'phase1'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(FIXTURE_RAW)


## 3. 加载 processed SROIE

如果 processed JSONL 不存在，先从 fixture raw 转换。

In [ ]:
from procureguard.extraction.datasets import iter_sroie_samples, read_processed_jsonl, write_processed_jsonl

if not FIXTURE_PROCESSED.exists():
    write_processed_jsonl(iter_sroie_samples(FIXTURE_RAW), FIXTURE_PROCESSED)
samples = read_processed_jsonl(FIXTURE_PROCESSED)
print('fixture smoke result, cannot be used as resume metric')
print('samples=', len(samples))


## 4. 展示 OCR tokens

这里读取的是 SROIE 标注里的 OCR token。真实图片 OCR smoke 可用 `scripts/phase1/ocr_regex_baseline.py` 手动执行。

In [ ]:
first = samples[0]
for token in first.tokens[:5]:
    print(token)


## 5. 运行 OCR + Regex baseline

baseline 输出 SROIE 四字段，每个字段包含 value、confidence、matched_text 和 rule_name。

In [ ]:
from procureguard.extraction.baseline import SroieRegexBaseline

baseline = SroieRegexBaseline()
extraction = baseline.extract(first.tokens)
for name, field in extraction.fields.items():
    print(name, field)


## 6. 输出 fixture baseline F1

这里是 fixture smoke result，不能作为真实模型效果。

In [ ]:
import time
from procureguard.extraction.metrics import build_evaluation_report, metrics_to_markdown

started_at = time.perf_counter()
predictions = [baseline.extract(sample.tokens).values() for sample in samples]
references = [sample.labels for sample in samples]
report = build_evaluation_report(
    baseline_name=baseline.baseline_name,
    predictions=predictions,
    references=references,
    sample_count=len(samples),
    data_source='tests/fixtures/sroie_minimal',
    is_fixture=True,
    started_at=started_at,
)
print(metrics_to_markdown(report))


## 7. 展示错误案例

错误分类是确定性规则，不调用 LLM。

In [ ]:
from procureguard.extraction.error_analysis import collect_error_cases, errors_to_markdown

errors = collect_error_cases([sample.sample_id for sample in samples], predictions, references)
print(errors_to_markdown(errors))


## 8. LayoutLMv3 Dataset / DataLoader 下一步

下一轮再接入真实 token 标签对齐、Dataset、DataLoader、训练循环、验证循环和 checkpoint。当前保留训练方向，不伪造真实数据集 F1。